# Training BERT on Kaggle

This notebook trains a BERT-base model implemented in PyTorch, including its embeddings, self-attention, encoder layers, MLM objective, and NSP objective. It uses CUDA when a Kaggle GPU is enabled and looks under `/kaggle/input` for the token sequences and tokenizer.

The notebook expects the datasets `bert-token-sequences` and `bert-tokenizer` to be attached as inputs. Enable the GPU in the notebook settings before running. Checkpoints are written to `/kaggle/working/checkpoints/bert_base_mlm/`; save a notebook version to keep the outputs.

In [1]:
import json
import random
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from tqdm.auto import tqdm

KAGGLE_INPUT_DIR = Path("/kaggle/input")
KAGGLE_WORKING_DIR = Path("/kaggle/working")


def find_data_file(filename):
    # Kaggle mounts attached datasets under /kaggle/input/<dataset-slug>/.
    if KAGGLE_INPUT_DIR.exists():
        matches = sorted(KAGGLE_INPUT_DIR.rglob(filename))
        if matches:
            return matches[0]
    local_path = Path(filename)
    if local_path.exists():
        return local_path
    raise FileNotFoundError(
        f"Could not find {filename}. Attach a Kaggle dataset containing it, "
        "or place it next to this notebook."
    )


DATA_FILE = find_data_file("bert_token_sequences.jsonl")
TOKENIZER_FILE = find_data_file("bert_tokenizer.json")
CHECKPOINT_ROOT = KAGGLE_WORKING_DIR if KAGGLE_WORKING_DIR.exists() else Path(".")
CHECKPOINT_DIR = CHECKPOINT_ROOT / "checkpoints/bert_base_mlm"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

with TOKENIZER_FILE.open(encoding="utf-8") as file:
    tokenizer_json = json.load(file)

# The WordPiece vocabulary maps token strings to integer IDs.
VOCAB = tokenizer_json["model"]["vocab"]
VOCAB_SIZE = len(VOCAB)
PAD_ID = VOCAB["[PAD]"]
CLS_ID = VOCAB["[CLS]"]
SEP_ID = VOCAB["[SEP]"]
MASK_ID = VOCAB["[MASK]"]
SPECIAL_TOKEN_IDS = {PAD_ID, CLS_ID, SEP_ID, MASK_ID}

# Use CUDA for multi-GPU training; keep XPU/CUDA/CPU fallbacks.
if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch, "xpu") and torch.xpu.is_available():
    device = torch.device("xpu")
else:
    device = torch.device("cpu")

print(f"PyTorch: {torch.__version__}")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"CUDA GPU: {torch.cuda.get_device_name(0)}")
print(f"Training sequences file: {DATA_FILE}")
print(f"Tokenizer file: {TOKENIZER_FILE}")
print(f"Checkpoints will be written to: {CHECKPOINT_DIR}")
print(f"Vocabulary size: {VOCAB_SIZE:,}")

PyTorch: 2.13.0+xpu
Device: xpu
Vocabulary size: 30,000


## Input data

The attached datasets provide `bert_token_sequences.jsonl` and `bert_tokenizer.json`. The code searches `/kaggle/input` for these files, so it doesn't depend on a particular mount path. The token file is read from disk by line offset; checkpoints go to `/kaggle/working`.

In [2]:
class TokenSequenceDataset(Dataset):
    def __init__(self, jsonl_path):
        self.jsonl_path = Path(jsonl_path)
        self.offsets = []
        self.file = None

        # Index lines without loading the complete corpus into memory.
        with self.jsonl_path.open("rb") as file:
            while True:
                offset = file.tell()
                line = file.readline()
                if not line:
                    break
                self.offsets.append(offset)

    def __len__(self):
        return len(self.offsets)

    def __getitem__(self, index):
        if self.file is None or self.file.closed:
            self.file = self.jsonl_path.open("rb")
        self.file.seek(self.offsets[index])
        record = json.loads(self.file.readline())
        return record["input_ids"]


class NSPPairDataset(Dataset):
    def __init__(self, token_dataset, indices):
        self.token_dataset = token_dataset
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def remove_outer_special_tokens(self, token_ids):
        # Source chunks were encoded as [CLS] content [SEP].
        if token_ids and token_ids[0] == CLS_ID:
            token_ids = token_ids[1:]
        if token_ids and token_ids[-1] == SEP_ID:
            token_ids = token_ids[:-1]
        return token_ids

    def __getitem__(self, item):
        first_index = self.indices[item]
        first_chunk = self.remove_outer_special_tokens(self.token_dataset[first_index])
        if len(first_chunk) < 2:
            first_chunk = first_chunk + [VOCAB.get("[UNK]", 1)] * (2 - len(first_chunk))

        split = random.randint(1, len(first_chunk) - 1)
        tokens_a = first_chunk[:split]
        is_next = random.random() < 0.5

        if is_next:
            tokens_b = first_chunk[split:]
            nsp_label = 1
        else:
            random_index = random.randrange(len(self.token_dataset))
            tokens_b = self.remove_outer_special_tokens(self.token_dataset[random_index])
            if not tokens_b:
                tokens_b = [VOCAB.get("[UNK]", 1)]
            nsp_label = 0

        # BERT input has three special tokens: [CLS], [SEP], [SEP].
        while len(tokens_a) + len(tokens_b) > MAX_LENGTH - 3:
            if len(tokens_a) > len(tokens_b):
                tokens_a.pop() if random.random() < 0.5 else tokens_a.pop(0)
            else:
                tokens_b.pop() if random.random() < 0.5 else tokens_b.pop(0)

        input_ids = [CLS_ID] + tokens_a + [SEP_ID] + tokens_b + [SEP_ID]
        token_type_ids = [0] * (len(tokens_a) + 2) + [1] * (len(tokens_b) + 1)
        return {
            "input_ids": input_ids,
            "token_type_ids": token_type_ids,
            "nsp_label": nsp_label,
        }


token_dataset = TokenSequenceDataset(DATA_FILE)
indices = list(range(len(token_dataset)))
random.Random(42).shuffle(indices)
validation_count = max(1, int(len(indices) * 0.01))
validation_indices = indices[:validation_count]
training_indices = indices[validation_count:]

print(f"Training sequences: {len(training_indices):,}")
print(f"Validation sequences: {len(validation_indices):,}")

Training sequences: 823,633
Validation sequences: 8,319


## Build training batches

Each batch is padded to its longest pair. MLM selects 15% of ordinary tokens: 80% become `[MASK]`, 10% are replaced with random tokens, and 10% stay unchanged. Segment IDs and the NSP label are included in each batch.

In [3]:
MAX_LENGTH = 512
MASK_PROBABILITY = 0.15
BATCH_SIZE = 4


def make_pretraining_batch(examples):
    batch_length = min(MAX_LENGTH, max(len(example["input_ids"]) for example in examples))
    input_ids = torch.full((len(examples), batch_length), PAD_ID, dtype=torch.long)
    token_type_ids = torch.zeros((len(examples), batch_length), dtype=torch.long)
    nsp_labels = torch.tensor([example["nsp_label"] for example in examples], dtype=torch.long)

    for row, example in enumerate(examples):
        ids = example["input_ids"][:batch_length]
        types = example["token_type_ids"][:batch_length]
        input_ids[row, :len(ids)] = torch.tensor(ids, dtype=torch.long)
        token_type_ids[row, :len(types)] = torch.tensor(types, dtype=torch.long)

    attention_mask = (input_ids != PAD_ID).long()
    mlm_labels = input_ids.clone()
    probability = torch.full(input_ids.shape, MASK_PROBABILITY)
    for special_id in SPECIAL_TOKEN_IDS:
        probability.masked_fill_(input_ids == special_id, 0.0)
    selected = torch.bernoulli(probability).bool()
    mlm_labels[~selected] = -100

    replace_with_mask = (torch.rand(input_ids.shape) < 0.80) & selected
    input_ids[replace_with_mask] = MASK_ID
    replace_with_random = (torch.rand(input_ids.shape) < 0.50) & selected & ~replace_with_mask
    random_tokens = torch.randint(VOCAB_SIZE, input_ids.shape, dtype=torch.long)
    input_ids[replace_with_random] = random_tokens[replace_with_random]

    return input_ids, token_type_ids, attention_mask, mlm_labels, nsp_labels


train_dataset = NSPPairDataset(token_dataset, training_indices)
validation_dataset = NSPPairDataset(token_dataset, validation_indices)
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=make_pretraining_batch,
    num_workers=0,
)
validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=make_pretraining_batch,
    num_workers=0,
)

## Model architecture

This is the BERT-base configuration: 12 encoder layers, 768 hidden units, 12 attention heads, and a 3,072-unit feed-forward layer. The model components are written directly in PyTorch.

In [4]:
HIDDEN_SIZE = 768
NUM_LAYERS = 12
NUM_HEADS = 12
INTERMEDIATE_SIZE = 3072
DROPOUT = 0.1


class BertEmbeddings(nn.Module):
    def __init__(self, vocab_size, hidden_size, max_length=512, type_vocab_size=2):
        super().__init__()
        self.word_embeddings = nn.Embedding(vocab_size, hidden_size, padding_idx=PAD_ID)
        self.position_embeddings = nn.Embedding(max_length, hidden_size)
        self.token_type_embeddings = nn.Embedding(type_vocab_size, hidden_size)
        self.layer_norm = nn.LayerNorm(hidden_size, eps=1e-12)
        self.dropout = nn.Dropout(DROPOUT)

    def forward(self, input_ids, token_type_ids):
        sequence_length = input_ids.shape[1]
        positions = torch.arange(sequence_length, device=input_ids.device).unsqueeze(0)

        embeddings = (
            self.word_embeddings(input_ids)
            + self.position_embeddings(positions)
            + self.token_type_embeddings(token_type_ids)
        )
        return self.dropout(self.layer_norm(embeddings))


class MultiHeadSelfAttention(nn.Module):
    def __init__(self, hidden_size, num_heads):
        super().__init__()
        if hidden_size % num_heads != 0:
            raise ValueError("hidden_size must be divisible by num_heads")
        self.num_heads = num_heads
        self.head_size = hidden_size // num_heads
        self.query = nn.Linear(hidden_size, hidden_size)
        self.key = nn.Linear(hidden_size, hidden_size)
        self.value = nn.Linear(hidden_size, hidden_size)
        self.output = nn.Linear(hidden_size, hidden_size)
        self.dropout = nn.Dropout(DROPOUT)

    def split_heads(self, tensor):
        batch_size, sequence_length, hidden_size = tensor.shape
        tensor = tensor.view(batch_size, sequence_length, self.num_heads, self.head_size)
        return tensor.transpose(1, 2)  # (batch, heads, sequence, head_size)

    def forward(self, hidden_states, attention_mask):
        query = self.split_heads(self.query(hidden_states))
        key = self.split_heads(self.key(hidden_states))
        value = self.split_heads(self.value(hidden_states))

        # Scaled dot-product attention: softmax(QK^T / sqrt(head_size))V
        scores = torch.matmul(query, key.transpose(-1, -2))
        scores = scores / (self.head_size ** 0.5)
        key_mask = attention_mask[:, None, None, :].bool()
        scores = scores.masked_fill(~key_mask, torch.finfo(scores.dtype).min)
        weights = self.dropout(torch.softmax(scores, dim=-1))

        context = torch.matmul(weights, value)
        context = context.transpose(1, 2).contiguous()
        batch_size, sequence_length, _, _ = context.shape
        context = context.view(batch_size, sequence_length, self.num_heads * self.head_size)
        return self.output(context)


class BertEncoderLayer(nn.Module):
    def __init__(self, hidden_size, num_heads, intermediate_size):
        super().__init__()
        self.attention = MultiHeadSelfAttention(hidden_size, num_heads)
        self.attention_norm = nn.LayerNorm(hidden_size, eps=1e-12)
        self.attention_dropout = nn.Dropout(DROPOUT)
        self.intermediate = nn.Linear(hidden_size, intermediate_size)
        self.output = nn.Linear(intermediate_size, hidden_size)
        self.output_norm = nn.LayerNorm(hidden_size, eps=1e-12)
        self.output_dropout = nn.Dropout(DROPOUT)

    def forward(self, hidden_states, attention_mask):
        attention_output = self.attention(hidden_states, attention_mask)
        hidden_states = self.attention_norm(hidden_states + self.attention_dropout(attention_output))

        feed_forward = F.gelu(self.intermediate(hidden_states))
        feed_forward = self.output(feed_forward)
        return self.output_norm(hidden_states + self.output_dropout(feed_forward))


class BertForMaskedLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embeddings = BertEmbeddings(vocab_size, HIDDEN_SIZE, MAX_LENGTH)
        self.encoder_layers = nn.ModuleList([
            BertEncoderLayer(HIDDEN_SIZE, NUM_HEADS, INTERMEDIATE_SIZE)
            for _ in range(NUM_LAYERS)
        ])
        self.mlm_dense = nn.Linear(HIDDEN_SIZE, HIDDEN_SIZE)
        self.mlm_norm = nn.LayerNorm(HIDDEN_SIZE, eps=1e-12)
        self.pooler = nn.Linear(HIDDEN_SIZE, HIDDEN_SIZE)
        self.nsp_classifier = nn.Linear(HIDDEN_SIZE, 2)
        # MLM decoder shares its weight matrix with the word embeddings.
        self.mlm_bias = nn.Parameter(torch.zeros(vocab_size))

        self.apply(self.initialize_weights)
        with torch.no_grad():
            self.embeddings.word_embeddings.weight[PAD_ID].zero_()

    @staticmethod
    def initialize_weights(module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.LayerNorm):
            nn.init.ones_(module.weight)
            nn.init.zeros_(module.bias)

    def forward(self, input_ids, token_type_ids, attention_mask):
        hidden_states = self.embeddings(input_ids, token_type_ids)
        for layer in self.encoder_layers:
            hidden_states = layer(hidden_states, attention_mask)
        hidden_states = F.gelu(self.mlm_dense(hidden_states))
        hidden_states = self.mlm_norm(hidden_states)
        # MLM predicts a token at every position; its loss ignores unmasked positions.
        mlm_logits = F.linear(hidden_states, self.embeddings.word_embeddings.weight, self.mlm_bias)
        # NSP classifies the pooled [CLS] representation as IsNext (1) or RandomNext (0).
        pooled = torch.tanh(self.pooler(hidden_states[:, 0]))
        nsp_logits = self.nsp_classifier(pooled)
        return mlm_logits, nsp_logits


model = BertForMaskedLanguageModel(VOCAB_SIZE).to(device)
if device.type == "cuda" and torch.cuda.device_count() >= 2:
    # DataParallel splits each batch across the first two CUDA devices.
    model = nn.DataParallel(model, device_ids=[0, 1])
    print("Using CUDA GPUs 0 and 1 in parallel.")
else:
    print(f"Using one device: {device}")
parameter_count = sum(parameter.numel() for parameter in model.parameters())
print(f"Parameters: {parameter_count:,}")

Parameters: 109,705,010


## Train and save checkpoints

The progress bars show the current epoch, training batches, and validation batches, with MLM and NSP losses. When two CUDA GPUs are available, `DataParallel` splits each batch between them. Gradient accumulation sets the effective batch size.

The notebook saves a latest checkpoint at the end of each epoch and updates `best_model.pt` whenever validation loss improves. Checkpoints are written atomically.

In [5]:
EPOCHS = 25
GRADIENT_ACCUMULATION_STEPS = 8
LEARNING_RATE = 1e-4
LOG_EVERY = 25

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    betas=(0.9, 0.999),
    eps=1e-8,
    weight_decay=0.01,
)


def calculate_losses(mlm_logits, nsp_logits, mlm_labels, nsp_labels):
    mlm_loss = F.cross_entropy(
        mlm_logits.reshape(-1, VOCAB_SIZE),
        mlm_labels.reshape(-1),
        ignore_index=-100,
    )
    nsp_loss = F.cross_entropy(nsp_logits, nsp_labels)
    return mlm_loss + nsp_loss, mlm_loss, nsp_loss


def evaluate(model, data_loader):
    model.eval()
    total_loss = 0.0
    total_mlm_loss = 0.0
    total_nsp_loss = 0.0
    total_batches = 0

    validation_bar = tqdm(data_loader, desc="Validation", unit="batch", leave=False)
    with torch.no_grad():
        for input_ids, token_type_ids, attention_mask, mlm_labels, nsp_labels in validation_bar:
            input_ids = input_ids.to(device)
            token_type_ids = token_type_ids.to(device)
            attention_mask = attention_mask.to(device)
            mlm_labels = mlm_labels.to(device)
            nsp_labels = nsp_labels.to(device)

            mlm_logits, nsp_logits = model(input_ids, token_type_ids, attention_mask)
            loss, mlm_loss, nsp_loss = calculate_losses(
                mlm_logits, nsp_logits, mlm_labels, nsp_labels
            )
            total_loss += loss.item()
            total_mlm_loss += mlm_loss.item()
            total_nsp_loss += nsp_loss.item()
            total_batches += 1
            validation_bar.set_postfix(
                total=f"{total_loss / total_batches:.4f}",
                mlm=f"{total_mlm_loss / total_batches:.4f}",
                nsp=f"{total_nsp_loss / total_batches:.4f}",
            )

    model.train()
    denominator = max(total_batches, 1)
    return (
        total_loss / denominator,
        total_mlm_loss / denominator,
        total_nsp_loss / denominator,
    )


def save_checkpoint_atomically(checkpoint, path):
    # Avoid leaving a half-written checkpoint if the cloud session stops during a save.
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    torch.save(checkpoint, temporary_path)
    temporary_path.replace(path)


best_validation_loss = float("inf")
best_checkpoint_path = CHECKPOINT_DIR / "best_model.pt"
latest_checkpoint_path = CHECKPOINT_DIR / "latest_checkpoint.pt"
epoch_bar = tqdm(range(1, EPOCHS + 1), desc="Training epochs", unit="epoch")

for epoch in epoch_bar:
    model.train()
    optimizer.zero_grad(set_to_none=True)
    running_total = 0.0
    running_mlm = 0.0
    running_nsp = 0.0
    running_batches = 0

    train_bar = tqdm(
        train_loader,
        desc=f"Train {epoch}/{EPOCHS}",
        unit="batch",
        leave=False,
    )
    for step, batch in enumerate(train_bar, start=1):
        input_ids, token_type_ids, attention_mask, mlm_labels, nsp_labels = batch
        input_ids = input_ids.to(device)
        token_type_ids = token_type_ids.to(device)
        attention_mask = attention_mask.to(device)
        mlm_labels = mlm_labels.to(device)
        nsp_labels = nsp_labels.to(device)

        mlm_logits, nsp_logits = model(input_ids, token_type_ids, attention_mask)
        loss, mlm_loss, nsp_loss = calculate_losses(
            mlm_logits, nsp_logits, mlm_labels, nsp_labels
        )
        (loss / GRADIENT_ACCUMULATION_STEPS).backward()
        running_total += loss.item()
        running_mlm += mlm_loss.item()
        running_nsp += nsp_loss.item()
        running_batches += 1

        should_update = (
            step % GRADIENT_ACCUMULATION_STEPS == 0
            or step == len(train_loader)
        )
        if should_update:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

        if step % LOG_EVERY == 0 or step == len(train_loader):
            train_bar.set_postfix(
                total=f"{running_total / running_batches:.4f}",
                mlm=f"{running_mlm / running_batches:.4f}",
                nsp=f"{running_nsp / running_batches:.4f}",
            )
            running_total = 0.0
            running_mlm = 0.0
            running_nsp = 0.0
            running_batches = 0

    validation_loss, validation_mlm_loss, validation_nsp_loss = evaluate(
        model, validation_loader
    )
    epoch_bar.set_postfix(
        val_total=f"{validation_loss:.4f}",
        val_mlm=f"{validation_mlm_loss:.4f}",
        val_nsp=f"{validation_nsp_loss:.4f}",
    )
    tqdm.write(
        f"Epoch {epoch}/{EPOCHS} | validation total={validation_loss:.4f} "
        f"(MLM={validation_mlm_loss:.4f}, NSP={validation_nsp_loss:.4f})"
    )

    model_to_save = model.module if isinstance(model, nn.DataParallel) else model
    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model_to_save.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "validation_loss": validation_loss,
        "validation_mlm_loss": validation_mlm_loss,
        "validation_nsp_loss": validation_nsp_loss,
        "config": {
            "vocab_size": VOCAB_SIZE,
            "hidden_size": HIDDEN_SIZE,
            "num_layers": NUM_LAYERS,
            "num_heads": NUM_HEADS,
            "intermediate_size": INTERMEDIATE_SIZE,
            "max_length": MAX_LENGTH,
        },
    }
    save_checkpoint_atomically(checkpoint, latest_checkpoint_path)

    if validation_loss < best_validation_loss:
        best_validation_loss = validation_loss
        best_model_checkpoint = {
            key: value for key, value in checkpoint.items()
            if key != "optimizer_state_dict"
        }
        save_checkpoint_atomically(best_model_checkpoint, best_checkpoint_path)
        tqdm.write(f"Saved best checkpoint (epoch {epoch}): {best_checkpoint_path}")

if not best_checkpoint_path.exists():
    raise RuntimeError("Training finished without saving a best-model checkpoint.")

best_checkpoint = torch.load(best_checkpoint_path, map_location="cpu")
print(
    f"Best checkpoint saved: {best_checkpoint_path.resolve()} "
    f"(epoch {best_checkpoint['epoch']}, validation loss {best_checkpoint['validation_loss']:.4f})"
)
print(f"Latest checkpoint: {latest_checkpoint_path.resolve()}")

  0%|          | 0/25 [00:00<?, ?it/s]

Epoch 1/25:   0%|          | 0/205909 [00:00<?, ?batch/s]

KeyboardInterrupt: 

## Load the best checkpoint

Use these lines in a later cell or notebook to restore the model:

In [ ]:
# checkpoint = torch.load(best_checkpoint_path, map_location=device)
# model = BertForMaskedLanguageModel(checkpoint["config"]["vocab_size"]).to(device)
# model.load_state_dict(checkpoint["model_state_dict"])
# model.eval()